# Behavioural Feature Extraction — RanSMAP Dataset

This notebook extracts **19 hand-crafted behavioural features** from raw hardware-level telemetry
collected by the RanSMAP framework (i3-gen12 machine, DDR4-2133-8G memory configuration).

**Input:** Raw per-operation CSVs (ATA disk I/O + memory access traces) per ransomware/benign class  
**Output:** `combined_i3-gen12_datasetF.parquet` — one row per 10-second window, 19 features + label

### Feature groups
| Group | Features |
|-------|----------|
| ATA disk I/O | read/write throughput, LBA variance, write entropy |
| Memory access | GPA variance (read/write/readwrite/exec), op counts, entropy |
| Memory page | 4KiB page access count, access_type counts (2 and 4) |

### Labelling
- **Benign:** `Idle`, `Firefox`, `Office`, `SDelete`, `AESCrypt`, `Zip`
- **Ransomware:** all other classes (label = 1)

## 1. Imports and Configuration

In [ ]:
import os
import glob
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────────
# Root directory containing one sub-folder per class (e.g. Idle/, WannaCry/, …)
ROOT = "/home/henrykabuye/Ransap/Datasets/RanSMAP/dataset/original/i3-gen12/ddr4-2133-8g"

# Where to write the final combined parquet
OUTPUT_PATH = "/mnt/4TBSTORE/Dr_Kabuye4TB/Batched/combined_i3-gen12_datasetF.parquet"

# ── Windowing ────────────────────────────────────────────────────────────────
# Each sample is a 10-second non-overlapping window of hardware events
WINDOW_SIZE = 10   # seconds
STEP_SIZE   = 10   # seconds (non-overlapping)

# ── Labelling ────────────────────────────────────────────────────────────────
BENIGN_CLASSES = {'Idle', 'Firefox', 'Office', 'SDelete', 'AESCrypt', 'Zip'}

# ── CSV schema ───────────────────────────────────────────────────────────────
# Column names for each raw telemetry file type
CSV_SCHEMA: Dict[str, List[str]] = {
    "ata_read.csv":      ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "ata_write.csv":     ["timestamp", "LBA", "size", "entropy", "flag1", "category"],
    "mem_exec.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_read.csv":      ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_readwrite.csv": ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
    "mem_write.csv":     ["timestamp", "GPA", "size", "MMIO_flag", "page_type", "access_type"],
}

# All 19 feature names produced by the extraction
ALL_FEATURE_KEYS = [
    "avg_read_throughput",
    "avg_write_throughput",
    "read_LBA_variance",
    "write_LBA_variance",
    "write_entropy_mean",
    "mem_read_entropy_mean",
    "mem_write_entropy_mean",
    "mem_readwrite_entropy_mean",
    "mem_read_GPA_variance",
    "mem_write_GPA_variance",
    "mem_readwrite_GPA_variance",
    "mem_exec_GPA_variance",
    "mem_read_count",
    "mem_write_count",
    "mem_readwrite_count",
    "mem_exec_count",
    "access_type_2_count",
    "access_type_4_count",
    "mem_4KiB_page_access_count",
]

print(f"ROOT       : {ROOT}")
print(f"OUTPUT     : {OUTPUT_PATH}")
print(f"Window     : {WINDOW_SIZE}s  Step: {STEP_SIZE}s")

## 2. Helper Functions

In [ ]:
def safe_var(series: pd.Series) -> float:
    """Variance that returns 0 on an empty or single-element series."""
    return float(series.var()) if len(series) > 1 else 0.0


def shannon_entropy(series: pd.Series) -> float:
    """
    Approximate information entropy of a numeric series.
    Uses -E[x * log2(x)] as a proxy (not a true probability distribution).
    Returns 0 for empty series.
    """
    if series.empty:
        return 0.0
    arr = series.to_numpy(dtype=float)
    return float(-(arr * np.log2(arr + 1e-10)).mean())


def build_windows(t_min: float, t_max: float) -> List[Tuple[float, float]]:
    """Return list of (start, end) window boundaries covering [t_min, t_max]."""
    starts = np.arange(t_min, t_max, STEP_SIZE)
    return [(s, s + WINDOW_SIZE) for s in starts]


def load_csv(path: str, fname: str) -> pd.DataFrame:
    """
    Load a telemetry CSV, assign the correct column names, coerce types,
    and drop rows with missing timestamps.
    """
    cols = CSV_SCHEMA[fname]
    df = pd.read_csv(path, header=None, names=cols, low_memory=False)
    df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])
    # Coerce all numeric columns
    for c in cols:
        if c != "timestamp":
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df


print("Helper functions defined.")

## 3. Per-Window Feature Extraction

For each 10-second window we compute:
- **ATA read/write:** throughput (bytes/s), LBA variance, write entropy
- **Memory ops:** GPA variance, operation count, size-based Shannon entropy
- **Memory page/access type:** access_type 2 & 4 counts, 4KiB page count

In [ ]:
def extract_features_from_file(
    file_path: str,
    fname: str,
    windows: List[Tuple[float, float]],
) -> Dict[float, Dict[str, float]]:
    """
    Given a single CSV file and a list of time windows, return a dict
    mapping window_start → {feature_name: value}.
    """
    df = load_csv(file_path, fname)
    result: Dict[float, Dict[str, float]] = {}

    for start, end in windows:
        win = df[(df["timestamp"] >= start) & (df["timestamp"] < end)]
        feats: Dict[str, float] = {}

        if fname == "ata_read.csv":
            feats["avg_read_throughput"] = float(win["size"].sum()) / WINDOW_SIZE
            feats["read_LBA_variance"]   = safe_var(win["LBA"])

        elif fname == "ata_write.csv":
            feats["avg_write_throughput"] = float(win["size"].sum()) / WINDOW_SIZE
            feats["write_LBA_variance"]   = safe_var(win["LBA"])
            feats["write_entropy_mean"]   = float(win["entropy"].mean()) if not win.empty else 0.0

        elif fname in ("mem_read.csv", "mem_write.csv", "mem_readwrite.csv", "mem_exec.csv"):
            # Derive operation label from filename: mem_read → read
            op = fname.replace("mem_", "").replace(".csv", "")  # read / write / readwrite / exec
            prefix = f"mem_{op}"

            feats[f"{prefix}_GPA_variance"] = safe_var(win["GPA"])
            feats[f"{prefix}_count"]         = float(len(win))

            # Entropy only for data-moving operations, not exec
            if op in ("read", "write", "readwrite"):
                feats[f"{prefix}_entropy_mean"] = shannon_entropy(win["size"])

            # access_type breakdown (shared across all mem files)
            if "access_type" in win.columns:
                feats["access_type_2_count"] = float((win["access_type"] == 2).sum())
                feats["access_type_4_count"] = float((win["access_type"] == 4).sum())

            # 4KiB page accesses: page_type value recorded as -0.0 in raw data
            if "page_type" in win.columns:
                feats["mem_4KiB_page_access_count"] = float((win["page_type"] == -0.0).sum())

        result[start] = feats

    return result


print("Feature extraction function defined.")

## 4. Process One Class Directory

Each class folder contains 6 CSV files. We process all of them, merge features
by window start time, fill any missing features with 0, and attach the binary label.

In [ ]:
def process_class_directory(class_dir: str) -> pd.DataFrame:
    """
    Process all 6 telemetry CSV files in a class directory.
    Returns a DataFrame where each row is one 10-second window.
    """
    class_name = Path(class_dir).name
    label = 0 if class_name in BENIGN_CLASSES else 1

    # Collect all timestamps across files to determine window boundaries
    all_timestamps: List[float] = []
    for fname in CSV_SCHEMA:
        fpath = os.path.join(class_dir, fname)
        if not os.path.exists(fpath):
            continue
        df_ts = pd.read_csv(fpath, header=None, usecols=[0])
        ts = pd.to_numeric(df_ts.iloc[:, 0], errors="coerce").dropna()
        all_timestamps.extend(ts.tolist())

    if not all_timestamps:
        print(f"  [WARN] No timestamps found in {class_name}, skipping.")
        return pd.DataFrame()

    windows = build_windows(min(all_timestamps), max(all_timestamps))

    # Extract features from each file and merge into a single dict keyed by window start
    merged: Dict[float, Dict[str, float]] = {w[0]: {} for w in windows}

    for fname in CSV_SCHEMA:
        fpath = os.path.join(class_dir, fname)
        if not os.path.exists(fpath):
            continue
        file_feats = extract_features_from_file(fpath, fname, windows)
        for start, feats in file_feats.items():
            merged[start].update(feats)

    # Build DataFrame, fill missing features with 0
    rows = []
    for start, feats in sorted(merged.items()):
        row = {"timestamp": start, "category": class_name, "label": label}
        for key in ALL_FEATURE_KEYS:
            row[key] = feats.get(key, 0.0)
        rows.append(row)

    return pd.DataFrame(rows)


print("Class directory processor defined.")

## 5. Run Extraction Across All Classes

In [ ]:
# Discover all class sub-directories
class_dirs = sorted([
    d for d in glob.glob(os.path.join(ROOT, "*"))
    if os.path.isdir(d)
])

print(f"Found {len(class_dirs)} class directories:")
for d in class_dirs:
    name = Path(d).name
    lbl  = "benign" if name in BENIGN_CLASSES else "ransomware"
    print(f"  {name:<20} [{lbl}]")

In [ ]:
# Process every class and collect results
all_frames: List[pd.DataFrame] = []

for class_dir in tqdm(class_dirs, desc="Classes"):
    class_name = Path(class_dir).name
    df_class = process_class_directory(class_dir)
    if df_class.empty:
        continue
    all_frames.append(df_class)
    print(f"  {class_name:<20}  windows={len(df_class):>6}  label={df_class['label'].iloc[0]}")

combined = pd.concat(all_frames, ignore_index=True)
print(f"\nTotal windows: {len(combined):,}")
print(f"Class distribution:\n{combined['category'].value_counts()}")
print(f"Label distribution:\n{combined['label'].value_counts()}")

## 6. Sanity Check

In [ ]:
print("Shape          :", combined.shape)
print("Columns        :", combined.columns.tolist())
print("\nMissing values :\n", combined.isnull().sum())
print("\nFeature stats  :")
combined[ALL_FEATURE_KEYS].describe().T

## 7. Save to Parquet

In [ ]:
# Ensure output directory exists
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

combined.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(combined):,} rows → {OUTPUT_PATH}")

# Quick reload verification
verify = pd.read_parquet(OUTPUT_PATH)
print(f"Reload check   : {verify.shape} — OK" if verify.shape == combined.shape else "[WARN] Shape mismatch on reload!")